In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.utils.class_weight import compute_sample_weight

from sklearn.ensemble import StackingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv('website_features.csv')
df_processed = df.iloc[:, 1:20]

target_col = df_processed.columns[0]
X = df_processed.drop(columns=[target_col])
y = df_processed[target_col]

if y.dtype == 'object':
    le = LabelEncoder()
    y = le.fit_transform(y)

X = pd.get_dummies(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=12)
X_pca = pca.fit_transform(X_scaled)

X_train, X_test, y_train, y_test = train_test_split(X_pca, y, test_size=0.2, random_state=20, stratify=y)
target_names = ['benign' ,'gpt generated',  'malicious']

GPT_train_data = X_train[y_train == 1]
benign_train_data = X_train[y_train == 0]
malicious_train_data = X_train[y_train == 2]
y_benign_only = y_train[y_train == 0]
y_gpt_only = y_train[y_train == 1]
y_malicious_only = y_train[y_train == 2]

# ========================= splitting larger classes into smaller subsets =========================


Benign1 = benign_train_data[0:len(benign_train_data)//3]
Benign2 = benign_train_data[len(benign_train_data)//3: 2*len(benign_train_data)//3]
Benign3 = benign_train_data[2*len(benign_train_data)//3:]
#mirror shape of above fill with y values
y_Benign1 = [0]*len(Benign1)
y_Benign2 = [0]*len(Benign2)
y_Benign3 = [0]*len(Benign3)

Malicious1 = malicious_train_data[0:len(malicious_train_data)//2]
Malicious2 = malicious_train_data[len(malicious_train_data)//2:]
y_Malicious1 = [2]*len(Malicious1)
y_Malicious2 = [2]*len(Malicious2)

# gpt not split
GPT_train_data = GPT_train_data
y_GPT = y_train[y_train == 1]



In [ ]:
pca = PCA(n_components=12)
X_pca = pca.fit_transform(X_scaled)

# X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=20, stratify=y)

In [ ]:
# ========================= Ensemble Method - Oversampling and Undersampling =========================
'''
Will create 6 models using unique samples for majority and reuse minority classes
'''

set1 = np.vstack([Benign1, GPT_train_data, Malicious1])
set2 = np.vstack([Benign2, GPT_train_data, Malicious1])
set3 = np.vstack([Benign3, GPT_train_data, Malicious1])
set4 = np.vstack([Benign1, GPT_train_data, Malicious2])
set5 = np.vstack([Benign2, GPT_train_data, Malicious2])
set6 = np.vstack([Benign3, GPT_train_data, Malicious2])

y_set1 = np.hstack([y_Benign1, y_GPT, y_Malicious1])
y_set2 = np.hstack([y_Benign2, y_GPT, y_Malicious1])
y_set3 = np.hstack([y_Benign3, y_GPT, y_Malicious1])
y_set4 = np.hstack([y_Benign1, y_GPT, y_Malicious2])
y_set5 = np.hstack([y_Benign2, y_GPT, y_Malicious2])
y_set6 = np.hstack([y_Benign3, y_GPT, y_Malicious2])

print(f'length of B3: {len(Benign3)} length of GPT_train: {len(GPT_train_data)} length of M2: {len(Malicious2)}')
print(f'length of yB3: {len(y_Benign3)} length of yGPT_train: {len(y_GPT)} length of yM2: {len(y_Malicious2)}')

model = RandomForestClassifier(n_estimators=30, random_state=20, class_weight='balanced', max_depth=20)

model1 = model.fit(set1, y_set1)
model2 = model.fit(set2, y_set2)
model3 = model.fit(set3, y_set3)
model4 = model.fit(set4, y_set4)
model5 = model.fit(set5, y_set5)
model6 = model.fit(set6, y_set6)

pred1 = model1.predict(X_test)
pred2 = model2.predict(X_test)
pred3 = model3.predict(X_test)
pred4 = model4.predict(X_test)
pred5 = model5.predict(X_test)
pred6 = model6.predict(X_test)

# ========================= Voting Mechanism =========================
# NOTE: Yes i could package as 6 tuple of array, but this is clearer for now
def voter(a,b,c,d,e,f):
    result = []
    for i in range(len(a)):
        votes = a[i], b[i], c[i], d[i], e[i], f[i]
        final_vote = max(set(votes), key=votes.count)
        result.append(final_vote)
    return result
    

predictions = voter(pred1, pred2, pred3, pred4, pred5, pred6)

print("\n--- Test Set Performance ---")
print(f"Accuracy: {accuracy_score(y_test, predictions):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, predictions, target_names=target_names))


length of B3: 213 length of GPT_train: 99 length of M2: 209
length of yB3: 730 length of yGPT_train: 99 length of yM2: 947


ValueError: Found input variables with inconsistent numbers of samples: [521, 1038]